# M2 CLV 수준·N/V 구성·가격 임베딩: test-only seed 42

검증 구간을 만들지 않습니다. DAY 1~697을 모두 학습하고, 고정 100 epoch의 마지막 checkpoint를 DAY 698~704 test에서 한 번만 평가합니다. DAY 705~711은 사용하지 않습니다. 이 노트북은 seed 42 기술적 비교용이며, 결과로 모형·수식·epoch·하이퍼파라미터를 다시 선택하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = '3c3dbe4d48269789e2c879361c0865d22f479713'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        text=True, capture_output=True,
    )
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_level_composition_price_test1 import (
    configure_m2_level_composition_price_test_run,
    preflight_summary,
    run_m2_level_composition_price_test,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_m2_level_composition_price_test_run(
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m2_level_composition_price_test_seed42_v1',
)
summary = preflight_summary(cfg)
assert cfg.seed == 42
assert summary['validation_constructed'] is False
assert summary['holdout_constructed'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_m2_level_composition_price_test(cfg)

In [ ]:
from IPython.display import display

print('1) seed 42 test 절대지표')
display(result_df)
print('2) matched rho=0·ID-only·CLV 순열 대조군 비교')
display(result_df.attrs['comparison'])
print('3) 서술적 판독 — 모형 재선택에 사용하지 않음')
print(json.dumps(result_df.attrs['descriptive_reading'], ensure_ascii=False, indent=2))
print('4) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))